# 02 — Model comparison results

After running `python -m src.training.train_all --models har garch xgb lstm transformer`,
this notebook loads the prediction CSVs and produces the comparison plots and tables
for the README and the paper.

**Outputs:**
1. Per-model QLIKE bar chart
2. Forecast-vs-realized line plots
3. Cumulative loss over time (crisis-window detection)
4. Scatter plots (Mincer-Zarnowitz visual check)
5. Diebold-Mariano significance table

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json
from pathlib import Path
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.viz.plots import (
    plot_forecast_vs_actual,
    plot_qlike_comparison,
    plot_scatter_forecast_vs_realized,
    plot_cumulative_loss_over_time,
)
from src.evaluation.metrics import diebold_mariano

## 1. Load all model prediction files

In [ ]:
results_dir = Path('../results/metrics')
models = ['har', 'garch', 'xgb', 'lstm', 'transformer']

predictions = {}
for m in models:
    path = results_dir / f'{m}_predictions.csv'
    if path.exists():
        predictions[m] = pd.read_csv(path, parse_dates=['date'])
        print(f'{m:>12}: {len(predictions[m]):>5} predictions')
    else:
        print(f'{m:>12}: MISSING ({path})')

summary = json.load(open(results_dir / 'all_models_summary.json'))

## 2. QLIKE comparison across models

In [ ]:
plot_qlike_comparison(summary, save_path='../results/figures/qlike_comparison.png')
plt.show()

## 3. Forecast-vs-realized for each model

In [ ]:
for name, preds in predictions.items():
    plot_forecast_vs_actual(preds, name, save_path=f'../results/figures/{name}_forecast.png')
    plt.show()

## 4. Cumulative loss over time

Spot crisis windows (2008-2009, COVID 2020, etc.) where models diverge.

In [ ]:
plot_cumulative_loss_over_time(predictions, save_path='../results/figures/cumulative_loss.png')
plt.show()

## 5. Diebold-Mariano significance table (vs HAR-RV)

In [ ]:
if 'har' in predictions:
    har_preds = predictions['har']
    rows = []
    for name, preds in predictions.items():
        if name == 'har':
            continue
        merged = har_preds[['date', 'y_true', 'y_pred']].rename(columns={'y_pred': 'y_pred_har'}).merge(
            preds[['date', 'y_pred']].rename(columns={'y_pred': 'y_pred_model'}), on='date'
        )
        e_har = (np.log(merged['y_pred_har'].clip(lower=1e-10)) - np.log(merged['y_true'].clip(lower=1e-10))) ** 2
        e_model = (np.log(merged['y_pred_model'].clip(lower=1e-10)) - np.log(merged['y_true'].clip(lower=1e-10))) ** 2
        dm, p = diebold_mariano(e_har.values, e_model.values, h=1)
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else 'ns'
        rows.append({'model': name, 'DM_stat': dm, 'p_value': p, 'sig': sig})
    print(pd.DataFrame(rows).to_string(index=False))

## 6. Final summary table for README

In [ ]:
rows = []
for name, res in summary.items():
    rows.append({
        'model': name,
        'QLIKE': res['qlike'],
        'MSE log-var': res['mse_log_var'],
        'N obs': res['n_observations'],
    })
print(pd.DataFrame(rows).to_string(index=False))